In [1]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.0 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving dataset.txt to dataset.txt


In [3]:
from datasets import load_dataset

dataset = load_dataset("text", data_files={"data": "dataset.txt"})
dataset

Generating data split: 0 examples [00:00, ? examples/s]

DatasetDict({
    data: Dataset({
        features: ['text'],
        num_rows: 59
    })
})

In [4]:
dataset = dataset["data"].train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 53
    })
    test: Dataset({
        features: ['text'],
        num_rows: 6
    })
})

In [5]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [7]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # ← changed from evaluation_strategy
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

In [11]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,nan
2,No log,nan
3,No log,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=42, training_loss=2.945400601341611, metrics={'train_runtime': 414.9666, 'train_samples_per_second': 0.383, 'train_steps_per_second': 0.101, 'total_flos': 10386358272000.0, 'train_loss': 2.945400601341611, 'epoch': 3.0})

In [12]:
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])

print(f"Perplexity: {perplexity}")

Perplexity: nan


In [13]:
trainer.save_model("fine_tuned_gpt2_ai_blog")
tokenizer.save_pretrained("fine_tuned_gpt2_ai_blog")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fine_tuned_gpt2_ai_blog/tokenizer_config.json',
 'fine_tuned_gpt2_ai_blog/tokenizer.json')

In [16]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre-wrap; }</style>"))

In [24]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre-wrap; }</style>"))

'''Fine-tuned GPT-2 Output'''
prompt = "Artificial Intelligence is transforming"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.2,
    do_sample=True,
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Artificial Intelligence is transforming industries, leading to faster productivity growth and better outcomes for customers. With AI systems that help automate complex tasks like customer service or finance management are enabling businesses smarter about how they spend their time rather than wasting valuable data on other activities.
 (Image: Pixabay/Thinkstock)


In [23]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre-wrap; }</style>"))

''' Original GPT-2 (before fine-tuning)'''
base_model = GPT2LMHeadModel.from_pretrained("gpt2")
base_model.to(model.device)

base_output = base_model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    do_sample=True,
)

print("Base GPT-2 Output:\n")
print(tokenizer.decode(base_output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Base GPT-2 Output:

Artificial Intelligence is transforming the way we interact with our digital world. Our brain is the first to recognize and process the natural world in a way that is unprecedented for any living thing.

Our brain is designed to recognize and process information in a way that is unprecedented for any living thing. Our ability to sense the world is enhanced by our ability to understand and interact with information.

The AI has developed a remarkable ability to sense and interact with information in an unprecedented way.

We can perceive and understand the world by the ability to understand and interact with information. We can be very sensitive to the changes in


In [25]:
trainer.save_model("final_gpt2_ai_blog_model")
tokenizer.save_pretrained("final_gpt2_ai_blog_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('final_gpt2_ai_blog_model/tokenizer_config.json',
 'final_gpt2_ai_blog_model/tokenizer.json')